# Step 3: Test non-representative sample of hard and soft cases

This notebook builds a curated non-representative test set for refining the annotation instructions. The goal is not to estimate prevalence in the full corpus. The goal is to create an informative set with hard cases, unclear cases, positive and negative crime-frame cases, and enough `NO_CRIME_FRAME` cases to reach `n = 500`.

The workflow is:

1. Load the outputs from Step 2.1 and Step 2.2.
2. Build a curated Step 3 test set with hard cases, unclear cases, `CRIME_FRAME_NEG`, `CRIME_FRAME_POS`, and `NO_CRIME_FRAME` filler cases.
3. Export a clean file for human annotation.
4. Import human labels and calculate F1 scores and Cohen's kappa.
5. Optionally test few-shot prompting on the same test set.

Important: This test set is intentionally enriched and non-representative. Do not use it to estimate how common crime framing is in the full dataset.


## Step 3.0: Imports and file paths

This section defines the file paths used in Step 3. It expects that Step 2 created the following CSV files:

- `results/crime_frame_neg.csv`
- `results/crime_frame_pos.csv`
- `results/unclear_frame.csv`
- `results/hard_cases_frame.csv`
- one or more full Step 2.1 result files named like `annotation_step2_1_*.csv`


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, f1_score, cohen_kappa_score

RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

# Input files from Step 2
CRIME_FRAME_NEG = RESULTS_DIR / "crime_frame_neg.csv"
CRIME_FRAME_POS = RESULTS_DIR / "crime_frame_pos.csv"
UNCLEAR_FRAME = RESULTS_DIR / "unclear_frame.csv"
HARD_CASES_FRAME = RESULTS_DIR / "hard_cases_frame.csv"

# Step 3 output files
STEP3_TESTSET_WITH_MODEL_LABELS = RESULTS_DIR / "step3_testset_with_model_labels.csv"
STEP3_TESTSET_FOR_HUMAN = RESULTS_DIR / "step3_testset_for_human_annotation.csv"
STEP3_HUMAN_COMPLETED = RESULTS_DIR / "step3_testset_human_completed.csv"
STEP3_EVALUATION = RESULTS_DIR / "step3_evaluation_summary.csv"
STEP3_FEWSHOT_RESULTS = RESULTS_DIR / "step3_fewshot_results.csv"

N_TESTSET = 500
NO_CRIME_FILL_SEED = 123

VALID_LABELS = ["NO_CRIME_FRAME", "CRIME_FRAME_NEG", "CRIME_FRAME_POS"]
print("Step 3 setup complete.")


## Step 3.1: Load the candidate files from Step 2

We load the separate case files from Step 2. Empty or missing files are handled safely so the notebook can still run if one category has not appeared yet.


In [ ]:
def read_optional_csv(path):
    if path.exists():
        df_loaded = pd.read_csv(path)
        print(f"Loaded {path}: {len(df_loaded)} rows")
        return df_loaded
    else:
        print(f"Missing {path}: using empty dataframe")
        return pd.DataFrame()

crime_neg = read_optional_csv(CRIME_FRAME_NEG)
crime_pos = read_optional_csv(CRIME_FRAME_POS)
unclear = read_optional_csv(UNCLEAR_FRAME)
hard_cases = read_optional_csv(HARD_CASES_FRAME)


## Step 3.2: Load full Step 2.1 outputs for `NO_CRIME_FRAME` filler cases

The separate case files contain the interesting candidates. To fill the rest of the 500 row test set, we use `NO_CRIME_FRAME` cases from the full Step 2.1 result files.


In [ ]:
step2_1_files = sorted(RESULTS_DIR.glob("annotation_step2_1_*.csv"))

if not step2_1_files:
    raise FileNotFoundError("No full Step 2.1 files found. Expected files like results/annotation_step2_1_*.csv")

step2_1_all = []
for file in step2_1_files:
    temp = pd.read_csv(file)
    temp["source_file"] = file.name
    step2_1_all.append(temp)

step2_1_all = pd.concat(step2_1_all, ignore_index=True)
step2_1_all = step2_1_all.drop_duplicates(subset=["article_id", "par_index"])

no_crime_candidates = step2_1_all[step2_1_all["label"] == "NO_CRIME_FRAME"].copy()

print(f"Loaded {len(step2_1_files)} Step 2.1 files")
print(f"Unique Step 2.1 rows: {len(step2_1_all)}")
print(f"NO_CRIME_FRAME filler candidates: {len(no_crime_candidates)}")


## Step 3.3: Standardize candidates and build the 500 row test set

The test set includes all available hard cases, unclear cases, negative crime-frame cases, and positive crime-frame cases. The remaining rows are filled with `NO_CRIME_FRAME` cases.

If the interesting cases exceed 500 rows, the notebook keeps a mixed sample of them and does not add filler rows.


In [ ]:
def standardize_cases(df_cases, case_source, model_label_col="label"):
    if df_cases is None or len(df_cases) == 0:
        return pd.DataFrame(columns=[
            "article_id", "par_index", "group", "text_block_en", "model_label", "case_source"
        ])

    df_cases = df_cases.copy()

    # Make sure there is one text column with the same name
    if "text_block_en" not in df_cases.columns and "text_block" in df_cases.columns:
        df_cases["text_block_en"] = df_cases["text_block"]

    # Hard case files usually use final_label instead of label
    if model_label_col not in df_cases.columns:
        if "final_label" in df_cases.columns:
            model_label_col = "final_label"
        else:
            df_cases["model_label"] = ""
            model_label_col = "model_label"

    keep_cols = ["article_id", "par_index", "group", "text_block_en"]
    existing_keep_cols = [c for c in keep_cols if c in df_cases.columns]

    out = df_cases[existing_keep_cols].copy()
    out["model_label"] = df_cases[model_label_col].values
    out["case_source"] = case_source

    # Keep useful Step 2 diagnostics if present
    for extra_col in ["agreement", "step2_1_label", "final_label"]:
        if extra_col in df_cases.columns:
            out[extra_col] = df_cases[extra_col].values

    return out

neg_cases = standardize_cases(crime_neg, "crime_frame_neg", model_label_col="label")
pos_cases = standardize_cases(crime_pos, "crime_frame_pos", model_label_col="label")
unclear_cases = standardize_cases(unclear, "unclear", model_label_col="label")
hard_case_rows = standardize_cases(hard_cases, "hard_case", model_label_col="final_label")
no_crime_rows = standardize_cases(no_crime_candidates, "no_crime_filler", model_label_col="label")

interesting_cases = pd.concat(
    [hard_case_rows, unclear_cases, neg_cases, pos_cases],
    ignore_index=True
)
interesting_cases = interesting_cases.drop_duplicates(subset=["article_id", "par_index"])

n_fill = N_TESTSET - len(interesting_cases)

if n_fill > 0:
    no_crime_fill = no_crime_rows.sample(
        n=min(n_fill, len(no_crime_rows)),
        random_state=NO_CRIME_FILL_SEED
    ).copy()
    step3_testset = pd.concat([interesting_cases, no_crime_fill], ignore_index=True)
else:
    print("More interesting cases than N_TESTSET. Sampling interesting cases down to N_TESTSET.")
    step3_testset = interesting_cases.sample(
        n=N_TESTSET,
        random_state=NO_CRIME_FILL_SEED
    ).copy()

step3_testset = step3_testset.drop_duplicates(subset=["article_id", "par_index"]).reset_index(drop=True)
step3_testset["testset_id"] = range(1, len(step3_testset) + 1)

# Put testset_id first
cols = ["testset_id"] + [c for c in step3_testset.columns if c != "testset_id"]
step3_testset = step3_testset[cols]

step3_testset.to_csv(STEP3_TESTSET_WITH_MODEL_LABELS, index=False)

print(f"Step 3 test set saved to {STEP3_TESTSET_WITH_MODEL_LABELS}")
print(f"Rows in Step 3 test set: {len(step3_testset)}")
print("
Case source distribution:")
print(step3_testset["case_source"].value_counts())
print("
Model label distribution:")
print(step3_testset["model_label"].value_counts())


## Step 3.4: Export a clean file for human annotation

This file is meant for manual or consensus annotation. It hides the model labels so that human annotators are not biased by the model output.

Fill the `human_label` column manually with exactly one of:

- `NO_CRIME_FRAME`
- `CRIME_FRAME_NEG`
- `CRIME_FRAME_POS`

Optional: use `human_notes` for short comments about difficult cases.


In [ ]:
human_export = step3_testset[["testset_id", "article_id", "par_index", "group", "text_block_en"]].copy()
human_export["human_label"] = ""
human_export["human_notes"] = ""

human_export.to_csv(STEP3_TESTSET_FOR_HUMAN, index=False)
print(f"Human annotation file saved to {STEP3_TESTSET_FOR_HUMAN}")
human_export.head()


## Step 3.5: Import human labels and evaluate model performance

After human or consensus annotation is finished, save the completed file as:

`results/step3_testset_human_completed.csv`

The file must include `testset_id` and `human_label`. The code below joins human labels to the model labels and calculates F1 scores and Cohen's kappa.


In [ ]:
if not STEP3_HUMAN_COMPLETED.exists():
    print(f"Human completed file not found yet: {STEP3_HUMAN_COMPLETED}")
    print("Fill the human_label column in the human annotation file, save it as step3_testset_human_completed.csv, then rerun this cell.")
else:
    human_completed = pd.read_csv(STEP3_HUMAN_COMPLETED)

    eval_df = step3_testset.merge(
        human_completed[["testset_id", "human_label"]],
        on="testset_id",
        how="left"
    )

    eval_df["human_label"] = eval_df["human_label"].astype(str).str.strip()
    eval_ready = eval_df[eval_df["human_label"].isin(VALID_LABELS)].copy()

    print(f"Rows with valid human labels: {len(eval_ready)} out of {len(eval_df)}")

    y_true = eval_ready["human_label"]
    y_pred = eval_ready["model_label"]

    print("
=== Confusion Matrix ===")
    cm = pd.DataFrame(
        confusion_matrix(y_true, y_pred, labels=VALID_LABELS),
        index=[f"human_{x}" for x in VALID_LABELS],
        columns=[f"model_{x}" for x in VALID_LABELS]
    )
    display(cm)

    print("
=== Classification Report ===")
    print(classification_report(y_true, y_pred, labels=VALID_LABELS, zero_division=0))

    macro_f1 = f1_score(y_true, y_pred, labels=VALID_LABELS, average="macro", zero_division=0)
    weighted_f1 = f1_score(y_true, y_pred, labels=VALID_LABELS, average="weighted", zero_division=0)
    kappa = cohen_kappa_score(y_true, y_pred, labels=VALID_LABELS)

    summary = pd.DataFrame({
        "metric": ["macro_f1", "weighted_f1", "cohen_kappa"],
        "value": [macro_f1, weighted_f1, kappa]
    })

    display(summary)
    summary.to_csv(STEP3_EVALUATION, index=False)
    print(f"Evaluation summary saved to {STEP3_EVALUATION}")


## Step 3.6: Inspect errors

This section helps identify where the instruction needs improvement. These rows are especially useful for revising the codebook or writing few-shot examples.


In [ ]:
if not STEP3_HUMAN_COMPLETED.exists():
    print("Human labels not available yet.")
else:
    errors = eval_ready[eval_ready["human_label"] != eval_ready["model_label"]].copy()
    print(f"Errors: {len(errors)}")
    display(errors[[
        "testset_id", "case_source", "group", "human_label", "model_label", "text_block_en"
    ]].head(30))


# Step 3.2: Few-shot examples

Few-shot examples should be manually curated. Ideally, do not copy cases from the Step 3 test set into the few-shot prompt, because that would bias the evaluation. Use invented examples or separate training examples.

Good few-shot examples should include:

- at least one clear case per label
- hard boundary cases, especially victim versus perpetrator
- cases where crime is mentioned but not linked to the group
- cases where terrorism or violence is legitimized by a group
- cases where security or prevention is framed positively

The order should be mixed so the model does not learn a simple label sequence.


In [ ]:
FEW_SHOT_EXAMPLES = [
    {
        "text": "Beispieltext hier einfügen.",
        "label": "NO_CRIME_FRAME",
        "explanation": "Keine explizite Verknüpfung der Gruppe mit Kriminalität oder Sicherheit."
    },
    {
        "text": "Beispieltext hier einfügen.",
        "label": "CRIME_FRAME_NEG",
        "explanation": "Die Gruppe wird explizit als Quelle von Gefahr, Kriminalität oder Sicherheitsbedrohung dargestellt."
    },
    {
        "text": "Beispieltext hier einfügen.",
        "label": "CRIME_FRAME_POS",
        "explanation": "Der Text rahmt Sicherheit, Prävention oder Schutz im Zusammenhang mit der Gruppe positiv."
    },
]

def format_few_shot_examples(examples):
    formatted = []
    for ex in examples:
        formatted.append(
            "Text:
"
            + ex["text"]
            + "

Label: "
            + ex["label"]
            + "
Explanation: "
            + ex["explanation"]
        )
    return "

---

".join(formatted)

few_shot_block = format_few_shot_examples(FEW_SHOT_EXAMPLES)
print(few_shot_block)


## Step 3.7: Optional few-shot annotation function

This version adds the few-shot examples to the prompt before the paragraph to annotate. It uses the same output format as the earlier annotation function.

Run this only after you have added real few-shot examples above and after the API variables are available from your setup.


In [ ]:
def annotate_few_shot(text, instructions, reminder, few_shot_examples, temperature=0.0):
    few_shot_text = format_few_shot_examples(few_shot_examples)

    prompt = (
        f"{instructions}

"
        f"Hier sind Beispiele für die Annotation:

{few_shot_text}

"
        f"Jetzt annotieren Sie diesen Absatz:

{text}

"
        f"{reminder}

"
        "Respond in this format exactly:
"
        "Label: <NO_CRIME_FRAME or CRIME_FRAME_NEG or CRIME_FRAME_POS>
"
        "Explanation: <one sentence explaining your choice>"
    )

    payload = {
        "model": MODEL,
        "temperature": temperature,
        "messages": [
            {"role": "system", "content": "Sie sind ein Experte für Inhaltsanalyse. Befolgen Sie die Annotationsanweisungen genau und antworten Sie immer im angegebenen Format."},
            {"role": "user", "content": prompt}
        ]
    }

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    for attempt in range(3):
        try:
            response = requests.post(
                f"{HOST}/chat/completions",
                json=payload,
                headers=headers,
                timeout=120
            )
            if response.status_code == 200:
                return response.json()["choices"][0]["message"]["content"].strip()
            else:
                print(f"  [attempt {attempt+1}] Status {response.status_code}")
        except Exception as e:
            print(f"  [attempt {attempt+1}] Error: {e}")
            time.sleep(5)

    return "ERROR"


## Step 3.8: Optional few-shot run on the Step 3 test set

This runs the few-shot prompt on the Step 3 test set and saves the result. It can then be compared against the human labels using the same metrics as above.


In [ ]:
RUN_FEW_SHOT = False

if RUN_FEW_SHOT:
    fewshot_results = []

    for i, row in step3_testset.iterrows():
        raw = annotate_few_shot(
            str(row["text_block_en"]),
            instructions,
            reminder,
            FEW_SHOT_EXAMPLES,
            temperature=0.0
        )
        label = parse_label(raw)
        explanation = parse_explanation(raw)

        fewshot_results.append({
            "testset_id": row["testset_id"],
            "article_id": row["article_id"],
            "par_index": row["par_index"],
            "group": row["group"],
            "text_block_en": row["text_block_en"],
            "fewshot_raw_output": raw,
            "fewshot_label": label,
            "fewshot_explanation": explanation,
        })

        if (i + 1) % 10 == 0:
            print(f"[{i+1}/{len(step3_testset)}] done")

    fewshot_results = pd.DataFrame(fewshot_results)
    fewshot_results.to_csv(STEP3_FEWSHOT_RESULTS, index=False)
    print(f"Few-shot results saved to {STEP3_FEWSHOT_RESULTS}")
else:
    print("RUN_FEW_SHOT is False. Set it to True when you want to run few-shot annotation.")


## Step 3.9: Optional evaluation of few-shot results

After few-shot annotation and human labels are available, this compares the few-shot labels to the human standard labels.


In [ ]:
if not STEP3_FEWSHOT_RESULTS.exists() or not STEP3_HUMAN_COMPLETED.exists():
    print("Few-shot results or human completed file not available yet.")
else:
    fewshot_results = pd.read_csv(STEP3_FEWSHOT_RESULTS)
    human_completed = pd.read_csv(STEP3_HUMAN_COMPLETED)

    few_eval = fewshot_results.merge(
        human_completed[["testset_id", "human_label"]],
        on="testset_id",
        how="left"
    )

    few_eval["human_label"] = few_eval["human_label"].astype(str).str.strip()
    few_eval_ready = few_eval[few_eval["human_label"].isin(VALID_LABELS)].copy()

    y_true = few_eval_ready["human_label"]
    y_pred = few_eval_ready["fewshot_label"]

    print("=== Few-shot Classification Report ===")
    print(classification_report(y_true, y_pred, labels=VALID_LABELS, zero_division=0))

    print("=== Few-shot Cohen's Kappa ===")
    print(cohen_kappa_score(y_true, y_pred, labels=VALID_LABELS))
